# OULAD: Data Quality: MART layer

Source: `` `ftw-week-07`.`03-mart` ``
Results: `` `ftw-week-07`.`01-raw`.dq_check_results `` under `layer = 'mart'`

Same framework as Raw and Clean: same table, same 15 columns, same status
logic so the existing dashboard covers all three layers with no changes.

## 1: Run context

In [0]:
%sql
DECLARE OR REPLACE VARIABLE dq_run_id STRING;
SET VARIABLE dq_run_id = uuid();

In [0]:
%sql
-- The results table already exists from the Raw suite. This is a no-op, kept so the notebook runs standalone against an empty store.
CREATE TABLE IF NOT EXISTS `ftw-week-07`.`01-raw`.dq_check_results (
    run_id        STRING,
    executed_at   TIMESTAMP,
    layer         STRING,
    dataset       STRING,
    check_name    STRING,
    check_type    STRING,
    status        STRING,
    severity      STRING,
    fail_count    BIGINT,
    total_count   BIGINT,
    fail_pct      DOUBLE,
    threshold_pct DOUBLE,
    metric_value  DOUBLE,
    owner         STRING,
    details       STRING
)
COMMENT 'One row per check per run. Append only. Shared across raw, clean, mart.';

## 2: Dimensions

Each dimension gets: surrogate-key uniqueness, natural-key uniqueness, no null
keys, and a row count reconciled against the Clean table it was built from.

The two key checks are separate on purpose. A surrogate key can be unique while
the natural key is duplicated, which would mean the dimension has silently
changed grain.

In [0]:
%sql
-- ========== dim_student_enrollment ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'mart' AND dataset = 'dim_student_enrollment';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`03-mart`.dim_student_enrollment
),
checks AS (
    SELECT 'unique_surrogate_key' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (SELECT student_enrollment_key FROM `ftw-week-07`.`03-mart`.dim_student_enrollment GROUP BY student_enrollment_key HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Duplicate PK fans out every fact join.' AS details
    UNION ALL
SELECT 'unique_natural_key', 'UNIQUE',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM (SELECT id_student, code_module, code_presentation FROM `ftw-week-07`.`03-mart`.dim_student_enrollment GROUP BY id_student, code_module, code_presentation HAVING COUNT(*) > 1)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Declared grain is the enrollment triplet. A duplicate here means the grain changed.'
    UNION ALL
SELECT 'not_null_keys', 'NOT_NULL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_student_enrollment WHERE student_enrollment_key IS NULL OR id_student IS NULL OR code_module IS NULL OR code_presentation IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Surrogate and natural keys must both be complete.'
    UNION ALL
SELECT 'rowcount_matches_clean', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT ABS((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_student_enrollment) - (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_info))) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Built 1:1 from clean.student_info. Any delta means rows were lost or duplicated.'
    UNION ALL
SELECT 'domain_final_result', 'DOMAIN',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_student_enrollment WHERE final_result NOT IN ('Distinction','Pass','Fail','Withdrawn')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Drives every outcome question. Drift here corrupts the dashboard silently.'
    UNION ALL
SELECT 'domain_imd_band_format', 'DOMAIN',
           'WARN', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_student_enrollment WHERE imd_band IS NOT NULL AND imd_band NOT IN ('0-10%','10-20%','20-30%','30-40%','40-50%','50-60%','60-70%','70-80%','80-90%','90-100%')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'EXPECTED TO FIRE. `10-20` ships without the % suffix and Clean does not normalise it.'
    UNION ALL
SELECT 'measure_distinct_students', 'MEASURE',
           'INFO', 0.0,
           CAST((SELECT 0) AS BIGINT),
           CAST((SELECT COUNT(DISTINCT id_student) FROM `ftw-week-07`.`03-mart`.dim_student_enrollment) AS DOUBLE),
           'Distinct students vs enrollments. The gap is students taking several presentations.'
    UNION ALL
SELECT 'measure_enrollments_with_zero_activity', 'MEASURE',
           'INFO', 0.0,
           CAST((SELECT 0) AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_student_enrollment se LEFT ANTI JOIN `ftw-week-07`.`03-mart`.fact_activity fa ON se.student_enrollment_key = fa.student_enrollment_key) AS DOUBLE),
           'Enrollments with NO rows in fact_activity. They are absent, not zero -- so AVG(total_clicks) silently excludes them. Engagement queries must start from this dimension and LEFT JOIN the fact.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'mart', 'dim_student_enrollment',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
-- ========== dim_assessment ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'mart' AND dataset = 'dim_assessment';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`03-mart`.dim_assessment
),
checks AS (
    SELECT 'unique_surrogate_key' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (SELECT assessment_key FROM `ftw-week-07`.`03-mart`.dim_assessment GROUP BY assessment_key HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Duplicate PK double-counts every submission.' AS details
    UNION ALL
SELECT 'unique_natural_key', 'UNIQUE',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM (SELECT id_assessment FROM `ftw-week-07`.`03-mart`.dim_assessment GROUP BY id_assessment HAVING COUNT(*) > 1)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Declared grain is one row per assessment.'
    UNION ALL
SELECT 'not_null_keys', 'NOT_NULL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_assessment WHERE assessment_key IS NULL OR id_assessment IS NULL OR code_module IS NULL OR code_presentation IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'code_module/code_presentation are load-bearing: fact_assessment resolves enrollment through them.'
    UNION ALL
SELECT 'rowcount_matches_clean', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT ABS((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_assessment) - (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.assessments))) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Built 1:1 from clean.assessments.'
    UNION ALL
SELECT 'domain_assessment_type', 'DOMAIN',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_assessment WHERE assessment_type NOT IN ('TMA','CMA','Exam')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Expected TMA, CMA or Exam.'
    UNION ALL
    SELECT 'measure_assessments_with_no_submissions', 'MEASURE',
           'INFO', 0.0, CAST(0 AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_assessment d
                 LEFT ANTI JOIN `ftw-week-07`.`03-mart`.fact_assessment f ON d.assessment_key = f.assessment_key) AS DOUBLE),
           'Assessments nobody submitted to. Not an error -- but they vanish from any INNER JOIN report, so any "assessments by type" count built off the fact will be short by this many.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'mart', 'dim_assessment',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
6,6


In [0]:
%sql
-- ========== dim_site ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'mart' AND dataset = 'dim_site';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`03-mart`.dim_site
),
checks AS (
    SELECT 'unique_surrogate_key' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (SELECT site_key FROM `ftw-week-07`.`03-mart`.dim_site GROUP BY site_key HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Duplicate PK double-counts clicks when fact_activity joins in.' AS details
    UNION ALL
SELECT 'unique_natural_key', 'UNIQUE',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM (SELECT id_site FROM `ftw-week-07`.`03-mart`.dim_site GROUP BY id_site HAVING COUNT(*) > 1)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'id_site is globally unique in the source -- 6,364 sites, 6,364 ids, none spanning presentations.'
    UNION ALL
SELECT 'not_null_keys', 'NOT_NULL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_site WHERE site_key IS NULL OR id_site IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'week_from/week_to excluded -- legitimately absent for always-on resources.'
    UNION ALL
SELECT 'rowcount_matches_clean', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT ABS((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_site) - (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.vle))) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Built 1:1 from clean.vle.'
    UNION ALL
SELECT 'domain_activity_type', 'DOMAIN',
           'WARN', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_site WHERE activity_type NOT IN ('dataplus','dualpane','externalquiz','folder','forumng','glossary','homepage','htmlactivity','oucollaborate','oucontent','ouelluminate','ouwiki','page','questionnaire','quiz','repeatactivity','resource','sharedsubpage','subpage','url')) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'A new activity type is news about the source, not corruption.'
    UNION ALL
    SELECT 'measure_sites_with_no_activity', 'MEASURE',
           'INFO', 0.0, CAST(0 AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_site d
                 LEFT ANTI JOIN `ftw-week-07`.`03-mart`.fact_activity f ON d.site_key = f.site_key) AS DOUBLE),
           'VLE resources nobody ever clicked. Same shape as the zero-activity enrollment measure: absent from the fact, not zero in it.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'mart', 'dim_site',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
6,6


## 3: Facts

Each fact gets: surrogate-key uniqueness, **declared-grain uniqueness**, FK
resolution, and a row count reconciled against its Clean source.

Grain uniqueness is the check that actually tests the model. A grain statement
in a comment is a claim; this is the test of it.

FK resolution is checked twice, on purpose. A NULL foreign key means the join
found nothing. A non-null key that matches no dimension row means the
dimensions were rebuilt after the facts, the surrogate keys were reassigned
and the facts now point at the wrong rows. The second failure is silent
everywhere else in the pipeline.

In [0]:
%sql
-- ========== fact_assessment ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'mart' AND dataset = 'fact_assessment';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`03-mart`.fact_assessment
),
checks AS (
    SELECT 'unique_surrogate_key' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (SELECT fact_assessment_key FROM `ftw-week-07`.`03-mart`.fact_assessment GROUP BY fact_assessment_key HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Duplicate PK.' AS details
    UNION ALL
SELECT 'unique_declared_grain', 'UNIQUE',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM (SELECT student_enrollment_key, assessment_key FROM `ftw-week-07`.`03-mart`.fact_assessment GROUP BY student_enrollment_key, assessment_key HAVING COUNT(*) > 1)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Declared grain: one row per enrollment per assessment. This is the test of that claim.'
    UNION ALL
SELECT 'fk_enrollment_not_null', 'NOT_NULL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_assessment WHERE student_enrollment_key IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'NULL FK means the join to dim_student_enrollment found nothing.'
    UNION ALL
SELECT 'fk_assessment_not_null', 'NOT_NULL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_assessment WHERE assessment_key IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'NULL FK means the join to dim_assessment found nothing.'
    UNION ALL
SELECT 'fk_enrollment_resolves', 'REFERENTIAL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_assessment f LEFT ANTI JOIN `ftw-week-07`.`03-mart`.dim_student_enrollment d ON f.student_enrollment_key = d.student_enrollment_key) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Non-null key pointing at no dimension row = dimensions rebuilt after the facts. Silent corruption.'
    UNION ALL
SELECT 'fk_assessment_resolves', 'REFERENTIAL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_assessment f LEFT ANTI JOIN `ftw-week-07`.`03-mart`.dim_assessment d ON f.assessment_key = d.assessment_key) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Same, against dim_assessment.'
    UNION ALL
SELECT 'rowcount_matches_clean', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT ABS((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_assessment) - (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_assessment))) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'INNER JOIN to dim_assessment could silently drop rows. This is what would catch it.'
    UNION ALL
SELECT 'reconcile_score_total', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT CASE WHEN ROUND((SELECT SUM(score) FROM `ftw-week-07`.`03-mart`.fact_assessment), 2) = ROUND((SELECT SUM(score) FROM `ftw-week-07`.`02-clean`.student_assessment), 2) THEN 0 ELSE 1 END) AS BIGINT),
           CAST((SELECT SUM(score) FROM `ftw-week-07`.`03-mart`.fact_assessment) AS DOUBLE),
           'Score total must survive the joins. Computed from Clean at runtime, not hardcoded.'
    UNION ALL
SELECT 'measure_banked_scores', 'MEASURE',
           'INFO', 0.0,
           CAST((SELECT 0) AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_assessment WHERE is_banked = 1) AS DOUBLE),
           'Scores carried from an earlier presentation, not earned here. Any average score by presentation that includes them is wrong.'
    UNION ALL
SELECT 'measure_null_scores', 'MEASURE',
           'INFO', 0.0,
           CAST((SELECT 0) AS BIGINT),
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_assessment WHERE score IS NULL) AS DOUBLE),
           'Non-submissions. AVG(score) ignores NULLs, so it silently excludes these. Decide whether a non-submission counts as a failure BEFORE quoting a failure rate.'
    UNION ALL
    SELECT 'conformance_module_agrees_both_paths', 'CONSISTENCY',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_assessment f
                 JOIN `ftw-week-07`.`03-mart`.dim_student_enrollment se ON f.student_enrollment_key = se.student_enrollment_key
                 JOIN `ftw-week-07`.`03-mart`.dim_assessment da ON f.assessment_key = da.assessment_key
                 WHERE se.code_module <> da.code_module
                    OR se.code_presentation <> da.code_presentation) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Same two-path risk via dim_assessment. This is also the join fact_assessment uses to resolve enrollment, so a non-zero here means the conformance path itself is broken.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'mart', 'fact_assessment',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
11,11


In [0]:
%sql
-- ========== fact_activity ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'mart' AND dataset = 'fact_activity';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`03-mart`.fact_activity
),
checks AS (
    SELECT 'unique_surrogate_key' AS check_name, 'UNIQUE' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT COUNT(*) FROM (SELECT fact_activity_key FROM `ftw-week-07`.`03-mart`.fact_activity GROUP BY fact_activity_key HAVING COUNT(*) > 1)) AS BIGINT) AS fail_count,
           CAST(NULL AS DOUBLE) AS metric_value,
           'Duplicate PK across 8.4M rows.' AS details
    UNION ALL
SELECT 'unique_declared_grain', 'UNIQUE',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM (SELECT student_enrollment_key, site_key, activity_date FROM `ftw-week-07`.`03-mart`.fact_activity GROUP BY student_enrollment_key, site_key, activity_date HAVING COUNT(*) > 1)) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Declared grain: one row per enrollment per site per day. Clean aggregated to this grain, so it must hold.'
    UNION ALL
SELECT 'fk_enrollment_not_null', 'NOT_NULL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_activity WHERE student_enrollment_key IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'NULL FK means the join to dim_student_enrollment found nothing.'
    UNION ALL
SELECT 'fk_site_not_null', 'NOT_NULL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_activity WHERE site_key IS NULL) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'NULL FK means the three-column join to dim_site found nothing.'
    UNION ALL
SELECT 'fk_enrollment_resolves', 'REFERENTIAL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_activity f LEFT ANTI JOIN `ftw-week-07`.`03-mart`.dim_student_enrollment d ON f.student_enrollment_key = d.student_enrollment_key) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Non-null key pointing at no dimension row = dimensions rebuilt after the facts.'
    UNION ALL
SELECT 'fk_site_resolves', 'REFERENTIAL',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_activity f LEFT ANTI JOIN `ftw-week-07`.`03-mart`.dim_site d ON f.site_key = d.site_key) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Same, against dim_site.'
    UNION ALL
SELECT 'rowcount_matches_clean', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT ABS((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_activity) - (SELECT COUNT(*) FROM `ftw-week-07`.`02-clean`.student_vle))) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Built 1:1 from clean.student_vle, which is already at daily grain.'
    UNION ALL
SELECT 'range_total_clicks_positive', 'RANGE',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_activity WHERE total_clicks IS NULL OR total_clicks <= 0) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'A logged interaction implies at least one click.'
    UNION ALL
SELECT 'reconcile_clicks_vs_clean', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT ABS((SELECT SUM(total_clicks) FROM `ftw-week-07`.`03-mart`.fact_activity) - (SELECT SUM(sum_click) FROM `ftw-week-07`.`02-clean`.student_vle))) AS BIGINT),
           CAST((SELECT SUM(total_clicks) FROM `ftw-week-07`.`03-mart`.fact_activity) AS DOUBLE),
           'Click total must survive the joins into the fact. Computed from Clean at runtime.'
    UNION ALL
    SELECT 'conformance_module_agrees_both_paths', 'CONSISTENCY',
           'FAIL', 0.0,
           CAST((SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_activity f
                 JOIN `ftw-week-07`.`03-mart`.dim_student_enrollment se ON f.student_enrollment_key = se.student_enrollment_key
                 JOIN `ftw-week-07`.`03-mart`.dim_site ds ON f.site_key = ds.site_key
                 WHERE se.code_module <> ds.code_module
                    OR se.code_presentation <> ds.code_presentation) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'code_module is reachable from this fact via TWO dimensions. GROUP BY se.code_module and GROUP BY ds.code_module must give the same answer. Non-zero = two dashboards on one model will disagree.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'mart', 'fact_activity',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
10,10


## 4: Cross-layer reconciliation

The checks above compare Mart against Clean. These compare Mart against **Raw**
— the source files: which is the stronger claim, because it survives the one
transformation where rows legitimately disappear (10,655,280 sessions collapsing
to 8,459,320 daily rows).

If `clicks_raw_to_mart` passes, one real measure reconciles across every layer
boundary in the pipeline. That is the number worth putting in the presentation.

In [0]:
%sql
-- ========== cross_layer ==========
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'mart' AND dataset = 'cross_layer';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH total AS (
    SELECT COUNT(*) AS n FROM `ftw-week-07`.`03-mart`.fact_activity
),
checks AS (
    SELECT 'clicks_raw_to_mart' AS check_name, 'RECONCILIATION' AS check_type,
           'FAIL' AS severity, 0.0 AS threshold_pct,
           CAST((SELECT ABS((SELECT SUM(TRY_CAST(sum_click AS BIGINT)) FROM `ftw-week-07`.`01-raw`.student_vle) - (SELECT SUM(total_clicks) FROM `ftw-week-07`.`03-mart`.fact_activity))) AS BIGINT) AS fail_count,
           CAST((SELECT SUM(TRY_CAST(sum_click AS BIGINT)) FROM `ftw-week-07`.`01-raw`.student_vle) AS DOUBLE) AS metric_value,
           'Total clicks, source file to star schema. Survives the session-to-daily aggregation.' AS details
    UNION ALL
SELECT 'enrollments_raw_to_mart', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT ABS((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_info) - (SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_student_enrollment))) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Every source enrollment reaches the dimension.'
    UNION ALL
SELECT 'submissions_raw_to_mart', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT ABS((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.student_assessment) - (SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.fact_assessment))) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Every source submission reaches the fact.'
    UNION ALL
SELECT 'assessments_raw_to_mart', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT ABS((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.assessments) - (SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_assessment))) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Every source assessment reaches the dimension.'
    UNION ALL
SELECT 'sites_raw_to_mart', 'RECONCILIATION',
           'FAIL', 0.0,
           CAST((SELECT ABS((SELECT COUNT(*) FROM `ftw-week-07`.`01-raw`.vle) - (SELECT COUNT(*) FROM `ftw-week-07`.`03-mart`.dim_site))) AS BIGINT),
           CAST(NULL AS DOUBLE),
           'Every source VLE resource reaches the dimension.'
    UNION ALL
SELECT 'measure_activity_day_range', 'MEASURE',
           'INFO', 0.0,
           CAST((SELECT 0) AS BIGINT),
           CAST((SELECT MIN(activity_date) FROM `ftw-week-07`.`03-mart`.fact_activity) AS DOUBLE),
           'Earliest activity day offset. Negative means pre-course access, which is valid and should not be floored to week 0 with CAST -- use FLOOR(activity_date / 7).'
    UNION ALL
    SELECT 'dim_course_exists', 'REFERENTIAL',
           'WARN', 0.0,
           CAST((SELECT CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END
                 FROM information_schema.tables
                 WHERE table_catalog = 'ftw-week-07' AND table_schema = '03-mart'
                   AND table_name = 'dim_course') AS BIGINT),
           CAST(NULL AS DOUBLE),
           'EXPECTED TO FIRE. clean.courses is unused, so module_presentation_length / presentation_year are unreachable. Q3 needs course length to compare modules of 234 vs 269 days. Also the fix for the two-path conformance risk above.'
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'mart', 'cross_layer',
       c.check_name, c.check_type,
       CASE WHEN c.severity = 'INFO' THEN 'INFO'
            WHEN c.fail_count = 0    THEN 'PASS'
            WHEN t.n > 0 AND c.fail_count / t.n <= c.threshold_pct THEN 'WARN'
            ELSE c.severity END,
       c.severity, c.fail_count, t.n,
       CASE WHEN t.n = 0 THEN NULL ELSE c.fail_count / t.n END,
       c.threshold_pct, c.metric_value, 'data-engineering', c.details
FROM checks c CROSS JOIN total t;

num_affected_rows
0


num_affected_rows,num_inserted_rows
7,7


## 5: Volume

In [0]:
%sql
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'mart' AND check_name = 'row_count_not_empty';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'mart', dataset,
       'row_count_not_empty', 'VOLUME',
       CASE WHEN n = 0 THEN 'FAIL' ELSE 'PASS' END, 'FAIL',
       CASE WHEN n = 0 THEN 1 ELSE 0 END, n,
       CASE WHEN n = 0 THEN 1.0 ELSE 0.0 END, 0.0,
       CAST(NULL AS DOUBLE), 'data-engineering',
       'Zero rows is a silent failure. Every other check passes vacuously on an empty table.'
FROM (
    SELECT dataset, MAX(total_count) AS n
    FROM `ftw-week-07`.`01-raw`.dq_check_results
    WHERE run_id = dq_run_id AND layer = 'mart' AND check_type <> 'VOLUME'
    GROUP BY dataset
);

num_affected_rows
0


num_affected_rows,num_inserted_rows
6,6


In [0]:
%sql
DELETE FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND layer = 'mart' AND check_name = 'volume_stable_vs_previous_run';

INSERT INTO `ftw-week-07`.`01-raw`.dq_check_results
WITH current_counts AS (
    SELECT dataset, MAX(total_count) AS n
    FROM `ftw-week-07`.`01-raw`.dq_check_results
    WHERE run_id = dq_run_id AND layer = 'mart' AND check_type <> 'VOLUME'
    GROUP BY dataset
),
previous_counts AS (
    SELECT dataset, total_count AS prev_n
    FROM (
        SELECT dataset, total_count,
               DENSE_RANK() OVER (PARTITION BY dataset ORDER BY executed_at DESC) AS rnk
        FROM `ftw-week-07`.`01-raw`.dq_check_results
        WHERE layer = 'mart' AND check_type <> 'VOLUME' AND run_id <> dq_run_id
    )
    WHERE rnk = 1
    GROUP BY dataset, total_count
)
SELECT dq_run_id, CURRENT_TIMESTAMP(), 'mart', c.dataset,
       'volume_stable_vs_previous_run', 'VOLUME',
       CASE WHEN p.prev_n IS NULL THEN 'INFO'
            WHEN ABS(c.n - p.prev_n) / p.prev_n <= 0.10 THEN 'PASS'
            ELSE 'WARN' END,
       'WARN',
       CASE WHEN p.prev_n IS NULL OR ABS(c.n - p.prev_n) / p.prev_n <= 0.10
            THEN 0 ELSE ABS(c.n - p.prev_n) END,
       c.n,
       CASE WHEN p.prev_n IS NULL THEN NULL ELSE ABS(c.n - p.prev_n) / p.prev_n END,
       0.10,
       CAST(p.prev_n AS DOUBLE),
       'data-engineering',
       CASE WHEN p.prev_n IS NULL
            THEN 'First run for this dataset. No baseline yet.'
            ELSE 'Row count vs previous run. metric_value holds the previous count.' END
FROM current_counts c
LEFT JOIN previous_counts p ON c.dataset = p.dataset;

num_affected_rows
0


num_affected_rows,num_inserted_rows
6,6


## 6: Exit gate

**Must return zero rows.** Anything here means the star schema does not hold and
no dashboard should be built on it.

In [0]:
%sql
SELECT dataset, check_name, check_type, fail_count, total_count,
       ROUND(fail_pct * 100, 4) AS fail_pct, details
FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND status = 'FAIL'
ORDER BY dataset, check_type;

dataset,check_name,check_type,fail_count,total_count,fail_pct,details


## 7: Summary

In [0]:
%sql
SELECT
    COUNT(*)                                                  AS checks_run,
    SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)          AS passed,
    SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END)          AS warnings,
    SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END)          AS failed,
    SUM(CASE WHEN status = 'INFO' THEN 1 ELSE 0 END)          AS measurements,
    ROUND(100.0 * SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN status <> 'INFO' THEN 1 ELSE 0 END), 0), 1) AS pass_rate_pct,
    CASE WHEN SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END) > 0 THEN 'STOP'
         WHEN SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END) > 0 THEN 'REVIEW'
         ELSE 'HEALTHY' END                                   AS overall
FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id;

checks_run,passed,warnings,failed,measurements,pass_rate_pct,overall
60,51,2,0,7,96.2,REVIEW


In [0]:
%sql
-- Open issues and measurements, worst first.
SELECT dataset, check_name, check_type, status, fail_count, total_count,
       ROUND(fail_pct * 100, 4) AS fail_pct, metric_value, details
FROM `ftw-week-07`.`01-raw`.dq_check_results
WHERE run_id = dq_run_id AND status IN ('FAIL','WARN','INFO')
ORDER BY CASE status WHEN 'FAIL' THEN 0 WHEN 'WARN' THEN 1 ELSE 2 END,
         fail_pct DESC NULLS LAST, dataset, check_name;

dataset,check_name,check_type,status,fail_count,total_count,fail_pct,metric_value,details
dim_student_enrollment,domain_imd_band_format,DOMAIN,WARN,3516,32593,10.7876,null,EXPECTED TO FIRE. `10-20` ships without the % suffix and Clean does not normalise it.
cross_layer,dim_course_exists,REFERENTIAL,WARN,1,8459320,0.0,null,"EXPECTED TO FIRE. clean.courses is unused, so module_presentation_length / presentation_year are unreachable. Q3 needs course length to compare modules of 234 vs 269 days. Also the fix for the two-path conformance risk above."
cross_layer,measure_activity_day_range,MEASURE,INFO,0,8459320,0.0,-25.0,"Earliest activity day offset. Negative means pre-course access, which is valid and should not be floored to week 0 with CAST -- use FLOOR(activity_date / 7)."
dim_assessment,measure_assessments_with_no_submissions,MEASURE,INFO,0,206,0.0,18.0,"Assessments nobody submitted to. Not an error -- but they vanish from any INNER JOIN report, so any ""assessments by type"" count built off the fact will be short by this many."
dim_site,measure_sites_with_no_activity,MEASURE,INFO,0,6364,0.0,96.0,"VLE resources nobody ever clicked. Same shape as the zero-activity enrollment measure: absent from the fact, not zero in it."
dim_student_enrollment,measure_distinct_students,MEASURE,INFO,0,32593,0.0,28785.0,Distinct students vs enrollments. The gap is students taking several presentations.
dim_student_enrollment,measure_enrollments_with_zero_activity,MEASURE,INFO,0,32593,0.0,3365.0,"Enrollments with NO rows in fact_activity. They are absent, not zero -- so AVG(total_clicks) silently excludes them. Engagement queries must start from this dimension and LEFT JOIN the fact."
fact_assessment,measure_banked_scores,MEASURE,INFO,0,173912,0.0,1909.0,"Scores carried from an earlier presentation, not earned here. Any average score by presentation that includes them is wrong."
fact_assessment,measure_null_scores,MEASURE,INFO,0,173912,0.0,173.0,"Non-submissions. AVG(score) ignores NULLs, so it silently excludes these. Decide whether a non-submission counts as a failure BEFORE quoting a failure rate."


In [0]:
%sql
-- All three layers side by side. Confirms one dashboard covers the pipeline.
SELECT layer,
       COUNT(*)                                                  AS checks_run,
       SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END)          AS passed,
       SUM(CASE WHEN status = 'WARN' THEN 1 ELSE 0 END)          AS warnings,
       SUM(CASE WHEN status = 'FAIL' THEN 1 ELSE 0 END)          AS failed,
       MAX(executed_at)                                          AS last_run
FROM `ftw-week-07`.`01-raw`.dq_check_results
GROUP BY layer
ORDER BY layer;

layer,checks_run,passed,warnings,failed,last_run
clean,241,185,22,26,2026-09-10T08:17:26.200Z
mart,225,192,5,0,2026-09-10T11:10:17.399Z
raw,219,172,15,0,2026-09-09T16:39:05.340Z
